In [ ]:
!pip install stable_baselines3

import gymnasium as gym
import numpy as np
import random
import math

from sympy import randprime
from stable_baselines3 import PPO

class FactorEnv(gym.Env):

    def __init__(self, max_steps=200):

        super().__init__()

        self.action_space = gym.spaces.Discrete(4)
        self.small_primes = [2,3,5,7,11,13,17,19,23,29]
        self.max_steps = max_steps

        self.observation_space = gym.spaces.Box(
            low=-1,
            high=1,
            shape=(13,),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        p = randprime(2**15, 2**16)
        q = randprime(2**15, 2**16)

        self.n = p * q
        self.remaining = self.n
        self.k = 2
        self.steps = 0

        return self._get_obs(), {}

    def step(self, action):

        reward = -0.1  # stronger time penalty

        # Action 0: trial division
        if action == 0:
            if self.remaining % self.k == 0:
                self.remaining //= self.k
                reward += 20

        # Action 1: increment divisor
        elif action == 1:
            self.k = min(self.k + 1, int(math.sqrt(self.n)))
            reward -= 0.01  # slight penalty for wasting moves

        # Action 2: Pollard's Rho
        elif action == 2:
            d = self._pollards_rho(self.remaining)

            if d not in [1, self.remaining]:
                self.remaining //= d
                reward += 20

        # Action 3: random reset of k
        elif action == 3:
            self.k = random.randint(2, 50)

        self.steps += 1

        terminated = self.remaining == 1
        truncated = self.steps >= self.max_steps

        if terminated:
            reward += 100

        if truncated:
            reward -= 5

        return self._get_obs(), reward, terminated, truncated, {}

    def _get_obs(self):

        log_n = math.log2(self.remaining) / 32
        k_norm = np.log2(self.k + 1) / 16  # improved scaling
        mods = [(self.remaining % p)/p for p in self.small_primes]
        step_frac = self.steps / self.max_steps

        return np.array(
            [log_n, k_norm] + mods + [step_frac],
            dtype=np.float32
        )

    def _pollards_rho(self, n, max_iter=50):

        if n % 2 == 0:
            return 2

        x = 2
        y = 2
        d = 1

        f = lambda x: (x**2 + 1) % n

        for _ in range(max_iter):
            x = f(x)
            y = f(f(y))
            d = math.gcd(abs(x - y), n)

            if d > 1:
                break

        return d


# Create environment
env = FactorEnv()

# Initialize PPO model
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    batch_size=64,
    n_steps=2048,
    gamma=0.99,
)

# Train model
model.learn(total_timesteps=100_000)

In [ ]:
#Single episode test
obs, _ = env.reset()
done = False

total_reward = 0

print(f"Initial n: {env.n}")

while not done:
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

    total_reward += reward

    print(f"Action: {action}, Remaining: {env.remaining}, k: {env.k}, Reward: {reward}")

print("Final remaining:", env.remaining)
print("Total reward:", total_reward)

In [ ]:
# Multiple episodes
def evaluate_model(env, model, episodes=100):

    success = 0
    total_steps = []
    total_rewards = []

    for _ in range(episodes):

        obs, _ = env.reset()
        done = False
        steps = 0
        ep_reward = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)

            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            steps += 1
            ep_reward += reward

        if env.remaining == 1:
            success += 1

        total_steps.append(steps)
        total_rewards.append(ep_reward)

    print("Success Rate:", success / episodes)
    print("Avg Steps:", np.mean(total_steps))
    print("Avg Reward:", np.mean(total_rewards))

evaluate_model(env, model, episodes=100)

In [ ]:
def trial_division_baseline(n, max_steps=200):

    k = 2
    steps = 0
    remaining = n

    while remaining > 1 and steps < max_steps:

        if remaining % k == 0:
            remaining //= k
        else:
            k += 1

        steps += 1

    success = (remaining == 1)
    return success, steps

In [ ]:
def evaluate_baseline(env, episodes=100):

    success = 0
    total_steps = []

    for _ in range(episodes):

        p = randprime(2**15, 2**16)
        q = randprime(2**15, 2**16)
        n = p * q

        s, steps = trial_division_baseline(n)

        if s:
            success += 1

        total_steps.append(steps)

    print("Baseline Success Rate:", success / episodes)
    print("Baseline Avg Steps:", np.mean(total_steps))

In [ ]:
print("=== RL MODEL ===")
evaluate_model(env, model)

print("\n=== BASELINE ===")
evaluate_baseline(env)

=== RL MODEL ===
Success Rate: 0.0
Avg Steps: 200.0
Avg Reward: -25.00000000000001

=== BASELINE ===
Baseline Success Rate: 0.0
Baseline Avg Steps: 200.0
